# Pareto: accuracy vs inference latency

Joins training **F1** (`results/ablation/*.json`, this package) with
inference **latency** (`donut/scripts/results/ablation/bench_speed.json`)
on **image size** — the knob that sets both. Pick the knee: the
smallest/fastest image that still hits your accuracy floor.

Run both sweeps first, with the **same** HxW values:
`donut/scripts/run_ablation.sh` and `scripts/exp_image_size.sh`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Quality: one JSON per fine-tuning run (this package).
Q_DIR = Path("../results/ablation")
# Speed: bench_speed.json from the donut inference package.
SPEED_JSON = Path("../../donut/scripts/results/ablation/bench_speed.json")

qrows = []
for p in sorted(Q_DIR.glob("*.json")):
    d = json.loads(p.read_text())
    if "f1_strict" not in d:
        continue
    h, w = d["image_size"]
    qrows.append(
        {"image_label": f"{h}x{w}", "image_px": h * w,
         "f1_strict": d["f1_strict"], "f1_soft": d["f1_soft"]}
    )
quality = pd.DataFrame(qrows)

speed = pd.read_json(SPEED_JSON) if False else None  # see next cell
print(f"quality runs: {len(quality)}; speed file exists: {SPEED_JSON.exists()}")
quality

In [ ]:
# Load inference speed records and reduce to one latency per image size.
# Pick the backend you intend to ship (default: fastest non-baseline at bs=1).
raw = json.loads(SPEED_JSON.read_text())
sp = pd.json_normalize(raw["records"])
sp["image_label"] = sp.image_height.astype(str) + "x" + sp.image_width.astype(str)
sp["image_px"] = sp.image_height * sp.image_width

SHIP_BATCH = 1
cand = sp[(sp.backend != "baseline") & (sp.batch_size == SHIP_BATCH)].copy()
# fastest backend per image size (lowest generate latency)
best = cand.loc[cand.groupby("image_label")["generate.mean_ms"].idxmin()]
speed = best[
    ["image_label", "image_px", "backend", "generate.mean_ms",
     "throughput.images_per_s"]
].rename(columns={"generate.mean_ms": "latency_ms", "throughput.images_per_s": "img_per_s"})
speed

## F1 vs inference latency

In [ ]:
# Join quality and speed on image size → F1 vs inference latency Pareto.
m = quality.merge(speed, on=["image_label", "image_px"], how="inner").sort_values("latency_ms")
if m.empty:
    print("No overlap. Use the SAME HxW values in exp_image_size.sh and run_ablation.sh.")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(m.latency_ms, m.f1_strict, marker="o")
    for _, r in m.iterrows():
        ax.annotate(
            f"{r.image_label}\n({r.backend})",
            (r.latency_ms, (r.f1_strict if r.f1_strict is not None else 0)),
            textcoords="offset points", xytext=(6, 6), fontsize=8,
        )
    ax.set_xlabel("inference latency  (generate ms, lower is better)")
    ax.set_ylabel("micro F1 strict (higher is better)")
    ax.set_title("Pareto: accuracy vs inference latency, by image size")
    ax.grid(alpha=0.3)
    plt.show()
    m

## Decision table

In [ ]:
# Decision: smallest/fastest image meeting an F1 floor.
F1_FLOOR = 0.90  # set to your acceptable accuracy
f1 = pd.to_numeric(m.f1_strict, errors="coerce").fillna(0.0)
ok = m[f1 >= F1_FLOOR].sort_values("latency_ms")
print(f"Configs with F1_strict >= {F1_FLOOR}:")
if ok.empty:
    print("  none yet — lower the floor or train larger images/longer.")
    m.sort_values("f1_strict", ascending=False)
else:
    print(f"  recommended: {ok.iloc[0].image_label} via {ok.iloc[0].backend} "
          f"({ok.iloc[0].latency_ms:.1f} ms, F1={ok.iloc[0].f1_strict:.3f})")
    ok